# 04 · Modeling
## Fresco Retailer Product Return Prediction
**Notebook:** 04 of 04  
**Prerequisite:** Run `03_feature_engineering.ipynb` first.  
**Goal:** Build, compare, tune, and finalize a production-ready classification pipeline.

### What this notebook covers
- Preprocessing pipeline (encoding + scaling + SMOTE-Tomek)
- Baseline pipeline test with Logistic Regression
- Multi-model comparison (DT, RF, XGBoost) using Pipline + Cross validation
- Hyperparameter tuning of best model (XGBoost)
- Save final production pipeline

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder , RobustScaler , StandardScaler , OrdinalEncoder
from category_encoders.binary import BinaryEncoder

from sklearn.model_selection import GridSearchCV, KFold, cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, recall_score, accuracy_score

from sklearn.linear_model import LogisticRegression

In [2]:
# ── Load final features from feature engineering notebook 03 ───────────────────────────────
df = pd.read_csv("data/03_features.csv")
print(f"Loaded features → shape: {df.shape}")
df.head(3)

Loaded features → shape: (19504, 13)


,product_category,Product_Subcategory,Quantity,Unit_Price,Price,Tax,Payment_mode,Reviews,Income,city,Return,total_price,tax_ratio
0,Books,Fiction,3,359,1077,188.475,Mobile Payments,1.0,67501.0,Hyderabad,1,1265.475,0.174838
1,Clothing,Women,5,1129,5645,592.725,Credit Card,1.0,102738.0,Bangalore,1,6237.725,0.104981
2,Home and kitchen,Bath,4,1327,5308,696.675,Debit Card,1.0,104013.0,Kolkata,1,6004.675,0.131225


# Modeling

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline


from sklearn.preprocessing import OneHotEncoder , RobustScaler , StandardScaler , OrdinalEncoder
from category_encoders.binary import BinaryEncoder

from sklearn.model_selection import GridSearchCV, KFold, cross_validate, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, recall_score, accuracy_score, precision_score, f1_score

from sklearn.linear_model import LogisticRegression

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

In [5]:
df_Cat = df.select_dtypes(include="object")

for col in df_Cat.columns :
    print(f"Column {col} has {df_Cat[col].nunique()} values")

Column product_category has 6 values
Column Product_Subcategory has 16 values
Column Payment_mode has 4 values
Column city has 10 values


In [6]:
df.describe()

,Quantity,Unit_Price,Price,Tax,Reviews,Income,Return,total_price,tax_ratio
count,19504.000000,19504.000000,19504.000000,19504.000000,19504.000000,19504.000000,19504.000000,19504.000000,19504.000000
mean,3.097724,776.168222,2393.920991,241.235454,3.882639,70516.888074,0.104953,2635.156445,0.103398
std,1.446259,414.249426,1795.395383,182.887380,1.414655,37418.384648,0.306500,1968.175145,0.026386
min,1.000000,70.000000,70.000000,7.350000,1.000000,7157.000000,0.000000,77.350000,0.020943
25%,2.000000,417.000000,945.000000,95.760000,3.000000,37950.000000,0.000000,1046.158750,0.104863
50%,3.000000,773.000000,1930.000000,190.785000,4.000000,69293.000000,0.000000,2126.020000,0.104937
75%,4.000000,1135.000000,3522.000000,349.965000,5.000000,99534.000000,0.000000,3875.235000,0.104968
max,5.000000,1500.000000,7500.000000,787.500000,5.000000,159984.000000,1.000000,8287.500000,0.524641


# 1- Pipeline building

In [7]:
Binary_encoding = ["Product_Subcategory", "city" ]

Label_encoding = ["product_category", "Payment_mode" ]

num_columns = ['Unit_Price', 'Price', 'Tax', 'Income', "tax_ratio", "total_price"]

In [ ]:
''''
Preprocessing Pipeline:
- cat: Apply Binary and Label encoding to categorical features 
- num_scaling: Apply RobustScaler to numeric features 
  → using RobustScaler becouse it handles outliers
- remainder='passthrough': keep any other columns unchanged.

'''

Transformation = ColumnTransformer(
    transformers=[
        ("Label_encoding", OneHotEncoder(sparse_output=False, drop="first"), Label_encoding),
        ("BE", BinaryEncoder(), Binary_encoding),
        ("num", RobustScaler(), num_columns)
    ],
    remainder="passthrough"
)


In [9]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## **2-Pipeline Testing**

- #### A base model will be trained using the preprocessing pipeline we created.  
- #### This test ensures that the pipeline is functioning properly before we evaluate multiple models.

In [ ]:
# Model Evaluation with Cross-Validation (testing to see every thing work or not)
steps = [
    ("Transform", Transformation),
    ("LR", LogisticRegression())
]

pipeline = Pipeline(steps=steps)

In [11]:
df.columns

Index(['product_category', 'Product_Subcategory', 'Quantity', 'Unit_Price',
       'Price', 'Tax', 'Payment_mode', 'Reviews', 'Income', 'city', 'Return',
       'total_price', 'tax_ratio'],
      dtype='object')

In [12]:
x = df.drop("Return", axis = 1 )
y = df["Return"]

In [13]:
kf = KFold(n_splits= 5, shuffle= True, random_state= 20 )
result = cross_validate(pipeline , x ,y , cv = kf , scoring="accuracy" , return_train_score= True)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

In [14]:
result["train_score"].mean()

0.9460110808589629

In [15]:
result["test_score"].mean()

0.9452421272651984

In [16]:
from sklearn.metrics import make_scorer, recall_score, accuracy_score, precision_score, f1_score
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_validate, KFold, RandomizedSearchCV
from imblearn.pipeline import Pipeline



from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB , MultinomialNB
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier

In [19]:
y.value_counts()

Return
0    17457
1     2047
Name: count, dtype: int64

In [ ]:
'''
We applied SMOTE to balance the minority class. After applying SMOTE, the code displays the number of instances in both classes.  

We will use these results to choose the best sampling ratio and integrate SMOTE into the final ML preprocessing pipeline.

'''

from collections import Counter

print("Before:", Counter(y))
X_res, y_res = SMOTETomek(smote=SMOTE(sampling_strategy={1: 9800}, random_state=24)).fit_resample(X, y)
print("After SMOTETomek:", Counter(y_res))

###### 9800 best number to put in pipeline 

Before: Counter({0: 17457, 1: 2047})
After SMOTETomek: Counter({0: 16768, 1: 9111})


## **3- Model Evaluation and Selection**

#### Several machine learning models will be trained and evaluated. The best performing model will then be selected, tuned, and deployed.

In [ ]:
# List of models to evaluate 
models = [
      ("DT", DecisionTreeClassifier()),
      ("RF", RandomForestClassifier()),
      ("XGB", XGBClassifier())
]

In [ ]:
"""
This block of code evaluates multiple machine learning models using a pipeline approach that apply:

1- Preprocessing : applies transformations we do

2- SMOTETomek : balances the dataset by oversampling the minority class

3- Cross-validation: ensures fair evaluation

4- Scoring metrics : measures accuracy, recall, precision, for both training and test sets.

5- Results : prints average scores so we can compare model performance and detect overfitting.

"""

scoring = {
    'recall': make_scorer(recall_score),
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'f1': make_scorer(f1_score)
}


for model in models:
    # pipeline with EncoTransformation, SMOTE Tomek and the models
    steps = [
       ("Transform", Transformation),
       ("SmoteTomek", SMOTETomek(smote=SMOTE(sampling_strategy={1: 9800}, random_state=24))),
       (model)
]

    pipeline = Pipeline(steps=steps)  # Apply pipeline steps

    # Cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
    result = cross_validate(pipeline, x, y, cv= skf, scoring= scoring, return_train_score= True)

    # Average recall
    print(f"{model[0]} Average Train Recall: {result['train_recall'].mean():.3}")
    print(f"{model[0]} Average Test Recall: {result['test_recall'].mean():.3}")

    # Average accuracy
    print(f"{model[0]} Average Train Accuracy: {result['train_accuracy'].mean():.3}")
    print(f"{model[0]} Average Test Accuracy: {result['test_accuracy'].mean():.3}")

    # Average precision
    print(f"{model[0]} Average Train Precision: {result['train_precision'].mean():.3}")
    print(f"{model[0]} Average Test Precision: {result['test_precision'].mean():.3}")
    
    # Average f1
    print(f"{model[0]} Average Train F1: {result['train_f1'].mean():.3}")
    print(f"{model[0]} Average Test F1: {result['test_f1'].mean():.3}\n")

DT Average Train Recall: 0.9960
DT Average Test Recall: 0.8632
DT Average Train Accuracy: 0.9994
DT Average Test Accuracy: 0.9644
DT Average Train Precision: 0.9982
DT Average Test Precision: 0.8100
DT Average Train F1: 0.9971
DT Average Test F1: 0.8357

RF Average Train Recall: 0.9957
RF Average Test Recall: 0.8583
RF Average Train Accuracy: 0.9994
RF Average Test Accuracy: 0.9748
RF Average Train Precision: 0.9987
RF Average Test Precision: 0.8971
RF Average Train F1: 0.9972
RF Average Test F1: 0.8772

XGB Average Train Recall: 0.9713
XGB Average Test Recall: 0.8481
XGB Average Train Accuracy: 0.9965
XGB Average Test Accuracy: 0.9754
XGB Average Train Precision: 0.9950
XGB Average Test Precision: 0.9118
XGB Average Train F1: 0.9830
XGB Average Test F1: 0.8787



# **XGBoost Model Selection**

XGBoost was selected as the best model. Some overfitting was observed, so we will optimize it by tuning the hyperparameters.

# 4- tuning

### **Perform hyperparameter tuning using RandomizedSearchCV.**

In [ ]:
# XGBoost transformation pipeline to tune 
steps = [
       ("Transform", Transformation),
       ("SmoteTomek", SMOTETomek(smote=SMOTE(sampling_strategy={1: 9800}, random_state=24))),
       ("XGB", XGBClassifier( ))
]

Final_pipeline= Pipeline(steps = steps)

In [ ]:
# parameter will used for RandomizedSearchCV
param_grid= {
                      "XGB__n_estimators": [250, 300, 400],
                      "XGB__max_depth": [5, 6, 8 ],
                      "XGB__learning_rate": [0.03, 0.04, 0.05 ],
                      "XGB__min_child_weight": [2, 3, 4, 5],
                      "XGB__gamma": [0, 1, 2]

}

In [ ]:
'''
# write documentation


'''
scorers = {
    'recall': make_scorer(recall_score),
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)

random_search = RandomizedSearchCV(
    estimator= Final_pipeline,
    param_distributions= param_grid,
    n_iter= 80,
    scoring= scorers,
    refit= 'recall',
    cv= skf,
    n_jobs= -1,
    random_state= 42,
    return_train_score= True
)

result = random_search.fit(x, y)


print(
    # Accuracy
    f"Train Acc: {result.cv_results_['mean_train_accuracy'][result.best_index_]:.4f} | "
    f"Test Acc: {result.cv_results_['mean_test_accuracy'][result.best_index_]:.4f} | "

    # Recall
    f"Train Recall: {result.cv_results_['mean_train_recall'][result.best_index_]:.4f} | "
    f"Test Recall: {result.cv_results_['mean_test_recall'][result.best_index_]:.4f} | "

    # Precision
    f"Train Precision: {result.cv_results_['mean_train_precision'][result.best_index_]:.4f} | "
    f"Test Precision: {result.cv_results_['mean_test_precision'][result.best_index_]:.4f}"
)


In [ ]:
# Print the best parameters of model from the RandomizedSearchCV 
result.best_params_

{'XGB__n_estimators': 250,
 'XGB__min_child_weight': 4,
 'XGB__max_depth': 5,
 'XGB__learning_rate': 0.03,
 'XGB__gamma': 1}

### **Final model with all things (preprocessing + Best model + tuning ) , ready to production**

In [168]:
# Set the pipeline parameters to the best parameters found from hyperparameter tuning
Final_pipeline.set_params(**result.best_params_)

Pipeline(steps=[('Transform',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Label_encoding',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['product_category',
                                                   'Payment_mode']),
                                                 ('BE', BinaryEncoder(),
                                                  ['Product_Subcategory',
                                                   'city']),
                                                 ('num', RobustScaler(),
                                                  ['Unit_Price', 'Price', 'Tax',
                                                   'Income', 'tax_ratio',
                                                   'total_price'])])),
                ('SmoteTomek',
                 SMOT...
                               feature_types=None, feature_weights=None,
                               gamma=1, grow_policy=None, importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None, min_child_weight=4,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=250,
                               n_jobs=None, num_parallel_tree=None, ...))])

In [ ]:
# Train the final pipeline with best model parameters and model and preprocessing 
XGBoost_Final_Model = Final_pipeline.fit(x, y)

In [ ]:
# import joblib

# joblib.dump(XGBoost_Final_Model, '/Users/mohammedmahmood/Desktop/Data projects/Projects/Data science/Supervised /Fresco Retailer Product Return Prediction/model/XGBoost_Final_Model.joblib')

['/Users/mohammedmahmood/Desktop/Data projects/Projects/Data science/Supervised /Fresco Retailer Product Return Prediction/model/XGBoost_Final_Model.joblib']

In [172]:
import sklearn
import category_encoders

print("scikit-learn:", sklearn.__version__)
print("category-encoders:", category_encoders.__version__)

scikit-learn: 1.3.2
category-encoders: 2.6.3
